## checking normality of features
If majority are normal will proceed with normal standardization, otherwise will use feature ranking to normalize

In [1]:


from scipy.stats import shapiro
import numpy as np
import pickle
import pandas as pd
from sklearn.preprocessing import QuantileTransformer


#importing data

try:
    with open('frameVectorAnnots.pkl', 'rb') as f:
        annots = pickle.load(f)
except:
    print("Error")

In [3]:
#stacking all frames into matrix
frameList = [] 

frameIds = [] 

for day in annots:
    for operation in annots[day]:
        for frame in annots[day][operation]:
            frameList.append(frame['frameVector'])
            frameIds.append((day, operation, frame['frameIdx']))


frameMatrix = np.array(frameList)
print(np.shape(frameMatrix))

#------converting matrix into pandas data frame--

#creating cols and rows
dfCols = []
dfRows = []

interpersonIdx = 1
tableFeatIdx = 1
for i in range(1, np.shape(frameMatrix)[1]+1):
    if i <=50:
        dfCols.append(f"Gpd_{i}")

    elif i == 51:
        dfCols.append(f"Num_clinicians")
    
    elif i>=52 and i<=55:
        dfCols.append(f"Table_feature_{tableFeatIdx}")
        tableFeatIdx+=1
    
    else: 
        dfCols.append(f"Interperson_feature_{interpersonIdx}")
        interpersonIdx+=1

for i in range(1, np.shape(frameMatrix)[0]+1):
    dfRows.append(f"Frame_{i}")

(626, 64)


In [4]:
print(dfCols)

['Gpd_1', 'Gpd_2', 'Gpd_3', 'Gpd_4', 'Gpd_5', 'Gpd_6', 'Gpd_7', 'Gpd_8', 'Gpd_9', 'Gpd_10', 'Gpd_11', 'Gpd_12', 'Gpd_13', 'Gpd_14', 'Gpd_15', 'Gpd_16', 'Gpd_17', 'Gpd_18', 'Gpd_19', 'Gpd_20', 'Gpd_21', 'Gpd_22', 'Gpd_23', 'Gpd_24', 'Gpd_25', 'Gpd_26', 'Gpd_27', 'Gpd_28', 'Gpd_29', 'Gpd_30', 'Gpd_31', 'Gpd_32', 'Gpd_33', 'Gpd_34', 'Gpd_35', 'Gpd_36', 'Gpd_37', 'Gpd_38', 'Gpd_39', 'Gpd_40', 'Gpd_41', 'Gpd_42', 'Gpd_43', 'Gpd_44', 'Gpd_45', 'Gpd_46', 'Gpd_47', 'Gpd_48', 'Gpd_49', 'Gpd_50', 'Num_clinicians', 'Table_feature_1', 'Table_feature_2', 'Table_feature_3', 'Table_feature_4', 'Interperson_feature_1', 'Interperson_feature_2', 'Interperson_feature_3', 'Interperson_feature_4', 'Interperson_feature_5', 'Interperson_feature_6', 'Interperson_feature_7', 'Interperson_feature_8', 'Interperson_feature_9']


In [5]:
frameVectorDf = pd.DataFrame(
    frameMatrix,
    columns = dfCols,
    index = dfRows
)

frameVectorDf.head()

,Gpd_1,Gpd_2,Gpd_3,Gpd_4,Gpd_5,Gpd_6,Gpd_7,Gpd_8,Gpd_9,Gpd_10,...,Table_feature_4,Interperson_feature_1,Interperson_feature_2,Interperson_feature_3,Interperson_feature_4,Interperson_feature_5,Interperson_feature_6,Interperson_feature_7,Interperson_feature_8,Interperson_feature_9
Frame_1,9.522396,9.745949,-2.441483,-2.424845,0.847117,-3.612987,-7.905068,-4.840728,7.682698,2.057512,...,1767.585850,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_2,-6.240740,7.566539,4.392485,-3.616573,8.635847,-11.345625,-5.310072,4.286849,0.439230,3.040247,...,2622.616756,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_3,-7.432102,-9.433077,-2.599083,-2.519187,6.199976,1.432799,-7.219241,7.671942,0.163997,3.052592,...,2445.534774,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_4,-10.756118,-3.976860,-1.695456,-4.020653,6.824022,-1.232486,-3.858355,-1.802732,6.092485,-1.278292,...,2456.068758,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_5,-9.633286,-4.889347,0.764947,-1.182092,8.913971,3.765080,-4.183628,3.302828,1.759320,2.955471,...,2400.928740,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# computes shapiro wilk for each in feature in the data frame
    # Ho: data is normally distributed vs Ha: data is non normally distributed
# returns a dataframe showing which features are normally distributed and which arent
def computeShapiro(df, alpha = 0.05):
    results = []
    rows  = [] 
    cols = ["Normality"] 
    for colName, colData in df.items():
        rows.append(colName)

        #checking normality
        _, p = shapiro(colData)
        normality = 0 if p<=alpha else 1   #1 if data is normal, 0 if not
        results.append(normality)

    featureDistrib = pd.DataFrame(
        results,
        columns = cols,
        index = rows
    )

    #compute percentage of normally distributed features
    normalFeatures = len(featureDistrib.query('Normality == 1'))

    return featureDistrib, normalFeatures

In [7]:
distrib, numNormal = computeShapiro(frameVectorDf)

print(f"{numNormal}/64 features are normally distributed")
distrib.head(64)


8/64 features are normally distributed


,Normality
Gpd_1,0
Gpd_2,1
Gpd_3,0
Gpd_4,1
Gpd_5,0
...,...
Interperson_feature_5,0
Interperson_feature_6,0
Interperson_feature_7,0
Interperson_feature_8,0


In [8]:
print(distrib.query('Normality == 1'))

        Normality
Gpd_2           1
Gpd_4           1
Gpd_11          1
Gpd_12          1
Gpd_15          1
Gpd_22          1
Gpd_24          1
Gpd_44          1


In [9]:
distrib, numNormal = computeShapiro(frameVectorDf, 0.01)

print(f"{numNormal}/64 features are normally distributed")

10/64 features are normally distributed


Majority of features are non normal, thus we will proceed with feature ranking

In [10]:
# maps output to uniform distribution [0-1]
# random_state controls small random noise to handle tied values , setting it ensures reproducibility
#default n_quantiles = 100 so we change it
qt = QuantileTransformer(output_distribution='uniform', n_quantiles=626, random_state=42) 
frameMatrixScaled = qt.fit_transform(frameMatrix)

In [11]:
##applying equal weighting to feature groups <-- this step can be swapped with ahp or anything else
w_gpd = 1/50
w_table = 1/4
w_inter = 1/9

gpdGrp = frameMatrixScaled[:, :50] * w_gpd
tableGrp = frameMatrixScaled[:, 50:55] * w_table
interpersonGrp = frameMatrixScaled[:, 55:] * w_inter
weightedFrames = np.concatenate([gpdGrp, tableGrp, interpersonGrp], axis=1)

In [12]:
print(np.shape(weightedFrames))

(626, 64)


In [13]:
#------converting matrix into pandas data frame--

#creating cols and rows
dfCols = []
dfRows = []

interpersonIdx = 1
tableFeatIdx = 1
for i in range(1, np.shape(weightedFrames)[1] + 1):  
    if i <=50:
        dfCols.append(f"Gpd_{i}")

    elif i == 51:
        dfCols.append(f"Num_clinicians")
    
    elif i>=52 and i<=55:
        dfCols.append(f"Table_feature_{tableFeatIdx}")
        tableFeatIdx+=1
    
    else: 
        dfCols.append(f"Interperson_feature_{interpersonIdx}")
        interpersonIdx+=1

for i in range(1, np.shape(weightedFrames)[0]+1):
    dfRows.append(f"Frame_{i}")


scaledFrameDf = pd.DataFrame(
    weightedFrames,
    columns = dfCols,
    index = dfRows
)

scaledFrameDf.head()

,Gpd_1,Gpd_2,Gpd_3,Gpd_4,Gpd_5,Gpd_6,Gpd_7,Gpd_8,Gpd_9,Gpd_10,...,Table_feature_4,Interperson_feature_1,Interperson_feature_2,Interperson_feature_3,Interperson_feature_4,Interperson_feature_5,Interperson_feature_6,Interperson_feature_7,Interperson_feature_8,Interperson_feature_9
Frame_1,0.016192,0.017760,0.006176,0.006720,0.011904,0.003840,0.000320,0.001728,0.019360,0.014464,...,0.0864,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_2,0.006112,0.016512,0.015680,0.004960,0.019040,0.000384,0.001920,0.017408,0.011776,0.016384,...,0.2104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_3,0.004928,0.001952,0.006112,0.006592,0.017920,0.013216,0.000576,0.019456,0.010944,0.016448,...,0.1660,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_4,0.001280,0.006240,0.006944,0.004384,0.018208,0.007584,0.003520,0.005856,0.018912,0.007360,...,0.1704,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_5,0.002176,0.005280,0.010400,0.008704,0.019136,0.016576,0.003200,0.016064,0.014848,0.016288,...,0.1540,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
print(scaledFrameDf.to_string())

              Gpd_1     Gpd_2     Gpd_3     Gpd_4     Gpd_5     Gpd_6     Gpd_7     Gpd_8     Gpd_9    Gpd_10    Gpd_11    Gpd_12    Gpd_13    Gpd_14    Gpd_15    Gpd_16    Gpd_17    Gpd_18    Gpd_19    Gpd_20    Gpd_21    Gpd_22    Gpd_23    Gpd_24    Gpd_25    Gpd_26    Gpd_27    Gpd_28    Gpd_29    Gpd_30    Gpd_31    Gpd_32    Gpd_33    Gpd_34    Gpd_35    Gpd_36    Gpd_37    Gpd_38    Gpd_39    Gpd_40    Gpd_41    Gpd_42    Gpd_43    Gpd_44    Gpd_45    Gpd_46    Gpd_47    Gpd_48    Gpd_49    Gpd_50  Num_clinicians  Table_feature_1  Table_feature_2  Table_feature_3  Table_feature_4  Interperson_feature_1  Interperson_feature_2  Interperson_feature_3  Interperson_feature_4  Interperson_feature_5  Interperson_feature_6  Interperson_feature_7  Interperson_feature_8  Interperson_feature_9
Frame_1    0.016192  0.017760  0.006176  0.006720  0.011904  0.003840  0.000320  0.001728  0.019360  0.014464  0.019360  0.002144  0.019168  0.009600  0.001088  0.013600  0.009376  0.013664  0.002080

## Regroup per operation

In [15]:
#group per operation again 
for day in annots:
    print(annots[day].keys())
    print(len(annots[day]))

dict_keys(['operation_6', 'operation_3', 'operation_9', 'operation_5', 'operation_8', 'operation_4', 'operation_7', 'operation_11', 'operation_2', 'operation_10', 'operation_1'])
11
dict_keys(['operation_1', 'operation_9', 'operation_2', 'operation_6', 'operation_7', 'operation_5', 'operation_8', 'operation_11', 'operation_4', 'operation_3', 'operation_10'])
11
dict_keys(['operation_1', 'operation_2', 'operation_3'])
3
dict_keys(['operation_1'])
1


In [16]:
for day in annots:
    for operation in annots[day]:
        print(f"{day} {operation}: {len(annots[day][operation])}")

day3 operation_6: 7
day3 operation_3: 23
day3 operation_9: 8
day3 operation_5: 33
day3 operation_8: 10
day3 operation_4: 21
day3 operation_7: 21
day3 operation_11: 15
day3 operation_2: 17
day3 operation_10: 13
day3 operation_1: 7
day2 operation_1: 105
day2 operation_9: 23
day2 operation_2: 97
day2 operation_6: 12
day2 operation_7: 7
day2 operation_5: 11
day2 operation_8: 11
day2 operation_11: 4
day2 operation_4: 8
day2 operation_3: 2
day2 operation_10: 2
day4 operation_1: 70
day4 operation_2: 41
day4 operation_3: 5
day1 operation_1: 53


In [17]:
OP_RANGES = [
    (0, 52), #day 1
    (53, 157), (158, 254), (257, 264), (265, 275), (276, 287), (288, 294), (295, 305), (306, 328), (331, 334), #day 2: dropped operations 3 (255,256) and 10 (329,330) because theyre only 2 frames long
    (335, 341), (342, 358), (359, 381), (382, 402), (403, 435), (436, 442), (443, 463), (464, 473), (474, 481), (482, 494), (495, 509), #day 3
    (510, 579), (580, 620), (621, 625) #day 4 
]

len(OP_RANGES)

24

In [18]:
print(type(scaledFrameDf))

<class 'pandas.core.frame.DataFrame'>


In [19]:
#converting pandas df to numpy arr
frames = scaledFrameDf.to_numpy()

In [20]:
operationVectors = []
for range in OP_RANGES:
    op = frames[range[0]:range[1]+1,]
    operationVectors.append(op)

operationVectors[0]

array([[0.016192  , 0.01776   , 0.006176  , ..., 0.        , 0.        ,
        0.        ],
       [0.006112  , 0.016512  , 0.01568   , ..., 0.        , 0.        ,
        0.        ],
       [0.004928  , 0.001952  , 0.006112  , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.002048  , 0.010688  , 0.013632  , ..., 0.07253333, 0.        ,
        0.08248889],
       [0.008736  , 0.004704  , 0.00544   , ..., 0.10506667, 0.        ,
        0.10897778],
       [0.015104  , 0.017696  , 0.005824  , ..., 0.06897778, 0.        ,
        0.07804444]], shape=(53, 64))

In [23]:
print(len(operationVectors))

24


In [22]:
print(type(operationVectors))

<class 'list'>


In [24]:
#exporting so we can work with operation list elsewhere

import pickle 

with open('operationList.pkl', 'wb') as f:
    pickle.dump(operationVectors, f)
    